# Project Technical Documentation AI Assistant
> Upload your own `.py` files and get instant AI documentation using **Llama 3**.

---

## Architecture Overview

```
Your Code Files (uploaded OR sample)
     │
     ▼
[1] AST Parser          ← Extracts functions, classes, docstrings
     │
     ▼
[2] Embedding Model     ← all-MiniLM-L6-v2 (SentenceTransformers)
     │
     ▼
[3] FAISS Vector Store  ← Stores semantic embeddings
     │
  [Query]
     │
     ▼
[4] RAG Retriever       ← Finds top-k relevant code chunks
     │
     ▼
[5] LLM (Llama 3 8B)    ← Generates professional documentation
     │
     ▼
[6] Gradio UI           ← Upload files + chat interface
```

**GPU Required:** Enable T4 GPU in Kaggle Settings → Accelerator

## Phase 1: Install Dependencies

In [1]:
!pip install -q transformers accelerate bitsandbytes sentence-transformers faiss-cpu gradio langchain
!pip install -q huggingface_hub
print("✅ All packages installed!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 31.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 66.1 MB/s eta 0:00:00:00:0100:01
✅ All packages installed!


## Phase 2: Sample Codebase (used as default until you upload your own)

In [2]:
import os

os.makedirs('sample_project', exist_ok=True)

with open('sample_project/auth.py', 'w') as f:
    f.write('''
import hashlib
import jwt
import datetime

SECRET_KEY = "my_secret_key"
TOKEN_EXPIRY_HOURS = 24

def hash_password(password: str) -> str:
    """Hashes a plain-text password using SHA-256."""
    return hashlib.sha256(password.encode()).hexdigest()

def login(username: str, password: str, db) -> dict:
    user = db.get_user(username)
    if not user:
        raise ValueError("User not found")
    if user["password"] != hash_password(password):
        raise ValueError("Invalid credentials")
    token = generate_token(user["id"])
    return {"status": "success", "token": token, "user_id": user["id"]}

def logout(session_id: str, cache) -> bool:
    if session_id in cache:
        cache.delete(session_id)
        return True
    return False

def generate_token(user_id: int) -> str:
    payload = {
        "user_id": user_id,
        "exp": datetime.datetime.utcnow() + datetime.timedelta(hours=TOKEN_EXPIRY_HOURS)
    }
    return jwt.encode(payload, SECRET_KEY, algorithm="HS256")

def verify_token(token: str) -> dict:
    try:
        return jwt.decode(token, SECRET_KEY, algorithms=["HS256"])
    except jwt.ExpiredSignatureError:
        raise ValueError("Token expired")
    except jwt.InvalidTokenError:
        raise ValueError("Invalid token")
''')

with open('sample_project/database.py', 'w') as f:
    f.write('''
import sqlite3
from typing import Optional, List

class DatabaseManager:
    """Manages SQLite database connections and CRUD operations."""

    def __init__(self, db_path: str):
        self.db_path = db_path
        self.connection = None

    def connect(self):
        self.connection = sqlite3.connect(self.db_path)
        self.connection.row_factory = sqlite3.Row

    def disconnect(self):
        if self.connection:
            self.connection.close()
            self.connection = None

    def execute_query(self, query: str, params: tuple = ()) -> List[dict]:
        cursor = self.connection.cursor()
        cursor.execute(query, params)
        return [dict(row) for row in cursor.fetchall()]

    def insert_user(self, username: str, email: str, hashed_password: str) -> int:
        cursor = self.connection.cursor()
        cursor.execute(
            "INSERT INTO users (username, email, password) VALUES (?, ?, ?)",
            (username, email, hashed_password)
        )
        self.connection.commit()
        return cursor.lastrowid

    def get_user(self, username: str) -> Optional[dict]:
        results = self.execute_query("SELECT * FROM users WHERE username = ?", (username,))
        return results[0] if results else None

    def delete_user(self, user_id: int) -> bool:
        cursor = self.connection.cursor()
        cursor.execute("DELETE FROM users WHERE id = ?", (user_id,))
        self.connection.commit()
        return cursor.rowcount > 0
''')

with open('sample_project/api.py', 'w') as f:
    f.write('''
from flask import Flask, request, jsonify
from functools import wraps

app = Flask(__name__)

def require_auth(f):
    @wraps(f)
    def decorated(*args, **kwargs):
        token = request.headers.get("Authorization", "").replace("Bearer ", "")
        if not token:
            return jsonify({"error": "Missing token"}), 401
        return f(*args, **kwargs)
    return decorated

def paginate_results(items: list, page: int, per_page: int = 20) -> dict:
    start = (page - 1) * per_page
    end = start + per_page
    return {
        "items": items[start:end],
        "total": len(items),
        "page": page,
        "pages": (len(items) + per_page - 1) // per_page
    }

@app.route("/api/v1/users", methods=["GET"])
@require_auth
def get_users():
    page = request.args.get("page", 1, type=int)
    users = []
    return jsonify(paginate_results(users, page))

@app.route("/api/v1/users/<int:user_id>", methods=["DELETE"])
@require_auth
def delete_user(user_id: int):
    return jsonify({"deleted": user_id})
''')

print("✅ Sample project created (auth.py, database.py, api.py)")
print("   You can upload your own .py files in the Gradio UI → Upload Your Code tab!")

✅ Sample project created (auth.py, database.py, api.py)
   You can upload your own .py files in the Gradio UI → Upload Your Code tab!


## Phase 3: AST Parser

In [3]:
import ast
import textwrap
from dataclasses import dataclass
from typing import List, Optional

@dataclass
class CodeChunk:
    chunk_type: str
    name: str
    source_code: str
    docstring: str
    file_path: str
    line_number: int
    args: List[str]
    parent_class: Optional[str] = None

    def to_text(self) -> str:
        parts = [
            f"File: {self.file_path}",
            f"Type: {self.chunk_type}",
            f"Name: {self.name}",
        ]
        if self.parent_class:
            parts.append(f"Class: {self.parent_class}")
        if self.args:
            parts.append(f"Arguments: {', '.join(self.args)}")
        if self.docstring:
            parts.append(f"Docstring: {self.docstring}")
        parts.append(f"Code:\n{self.source_code}")
        return "\n".join(parts)


class ASTCodeParser:
    def parse_file(self, filepath: str) -> List[CodeChunk]:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            source = f.read()
        try:
            tree = ast.parse(source)
        except SyntaxError as e:
            print(f"⚠️ Syntax error in {filepath}: {e}")
            return []
        chunks = []
        lines = source.splitlines()
        for node in ast.walk(tree):
            if isinstance(node, ast.FunctionDef):
                chunks.append(self._extract_function(node, lines, filepath))
            elif isinstance(node, ast.ClassDef):
                chunks.append(self._extract_class(node, lines, filepath))
                for item in node.body:
                    if isinstance(item, ast.FunctionDef):
                        chunks.append(self._extract_function(item, lines, filepath, parent_class=node.name))
        return chunks

    def _extract_function(self, node, lines, filepath, parent_class=None) -> CodeChunk:
        start = node.lineno - 1
        end = node.end_lineno
        source = "\n".join(lines[start:end])
        args = [arg.arg for arg in node.args.args if arg.arg != 'self']
        return CodeChunk(
            chunk_type='method' if parent_class else 'function',
            name=node.name,
            source_code=textwrap.dedent(source),
            docstring=ast.get_docstring(node) or "",
            file_path=filepath,
            line_number=node.lineno,
            args=args,
            parent_class=parent_class
        )

    def _extract_class(self, node, lines, filepath) -> CodeChunk:
        start = node.lineno - 1
        end = node.end_lineno
        source = "\n".join(lines[start:end])
        return CodeChunk(
            chunk_type='class',
            name=node.name,
            source_code=textwrap.dedent(source),
            docstring=ast.get_docstring(node) or "",
            file_path=filepath,
            line_number=node.lineno,
            args=[]
        )

    def parse_directory(self, directory: str) -> List[CodeChunk]:
        all_chunks = []
        for root, _, files in os.walk(directory):
            for file in files:
                if file.endswith('.py'):
                    chunks = self.parse_file(os.path.join(root, file))
                    all_chunks.extend(chunks)
        return all_chunks


parser = ASTCodeParser()
chunks = parser.parse_directory('sample_project')
print(f"✅ Parsed {len(chunks)} chunks from sample project")

✅ Parsed 25 chunks from sample project


## Phase 4: Embeddings + FAISS

In [4]:
import numpy as np
import faiss
from sentence_transformers import SentenceTransformer

print("⏳ Loading embedding model...")
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model ready!")


class FAISSVectorStore:
    def __init__(self, embedding_model):
        self.model = embedding_model
        self.index = None
        self.chunks = []
        self.texts = []
        self.dimension = 384

    def build_index(self, chunks: List[CodeChunk]):
        self.chunks = chunks
        self.texts = [c.to_text() for c in chunks]
        print(f"⏳ Embedding {len(self.texts)} chunks...")
        embeddings = self.model.encode(self.texts, show_progress_bar=True,
                                       batch_size=32, convert_to_numpy=True)
        faiss.normalize_L2(embeddings)
        self.index = faiss.IndexFlatIP(self.dimension)
        self.index.add(embeddings.astype('float32'))
        print(f"✅ FAISS index built with {self.index.ntotal} vectors!")

    def search(self, query: str, top_k: int = 5) -> List[dict]:
        if not self.index or self.index.ntotal == 0:
            return []
        query_vec = self.model.encode([query], convert_to_numpy=True)
        faiss.normalize_L2(query_vec)
        scores, indices = self.index.search(query_vec.astype('float32'), top_k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            if idx != -1:
                results.append({'chunk': self.chunks[idx], 'score': float(score), 'text': self.texts[idx]})
        return results


vector_store = FAISSVectorStore(embedding_model)
vector_store.build_index(chunks)
print("✅ Vector store ready!")

⏳ Loading embedding model...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ Embedding model ready!
⏳ Embedding 25 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS index built with 25 vectors!
✅ Vector store ready!


## Phase 5: Load LLM

In [12]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline
import requests
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")

USE_LOCAL_LLM = torch.cuda.is_available()
llm_pipeline = None
tokenizer = None

print(f"GPU Available: {USE_LOCAL_LLM}")
if USE_LOCAL_LLM:
    print(f"GPU: {torch.cuda.get_device_name(0)}")

# ── Llama 3 8B Instruct ─────────────────────────────────────────────────────
# NOTE: Llama 3 is a gated model on HuggingFace.
# You must: 1) Accept Meta's license at https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct
#           2) Add your HF_TOKEN to Kaggle Secrets (Settings → Add-ons → Secrets)
# ---------------------------------------------------------------------------
MODEL_ID = "meta-llama/Meta-Llama-3-8B-Instruct"

if USE_LOCAL_LLM:
    print(f"⏳ Loading {MODEL_ID} with 4-bit quantization...")
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.bfloat16
    )
    # Pass HF_TOKEN for gated model access
    token_arg = HF_TOKEN if HF_TOKEN else True
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=token_arg)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        token=token_arg
    )
    tokenizer.pad_token = tokenizer.eos_token
    llm_pipeline = pipeline(
        "text-generation",
        model=model,
        tokenizer=tokenizer,
        max_new_tokens=512,
        do_sample=True,
        temperature=0.1,
        top_p=0.9,
        repetition_penalty=1.1,
        return_full_text=False   # Only return newly generated text, not the prompt
    )
    print("✅ Llama 3 8B Instruct loaded!")
else:
    print("⚠️ No GPU found. Will use HF Inference API fallback if HF_TOKEN is set.")


def hf_inference_generate(prompt: str) -> str:
    """Calls HuggingFace Inference API for Llama 3 (no-GPU fallback)."""
    if not HF_TOKEN:
        return None
    API_URL = "https://api-inference.huggingface.co/models/meta-llama/Meta-Llama-3-8B-Instruct"
    headers = {"Authorization": f"Bearer {HF_TOKEN}"}
    payload = {
        "inputs": prompt,
        "parameters": {"max_new_tokens": 512, "temperature": 0.1, "return_full_text": False}
    }
    response = requests.post(API_URL, headers=headers, json=payload)
    result = response.json()
    if isinstance(result, list):
        return result[0].get("generated_text", "").strip()
    return None

GPU Available: True
GPU: Tesla T4
⏳ Loading meta-llama/Meta-Llama-3-8B-Instruct with 4-bit quantization...


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

Passing `generation_config` together with generation-related arguments=({'temperature', 'top_p', 'repetition_penalty', 'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


✅ Llama 3 8B Instruct loaded!


## Phase 6: RAG Assistant

In [13]:
class TechDocAssistant:
    def __init__(self, vector_store, llm_pipeline=None):
        self.vs = vector_store
        self.llm = llm_pipeline

    def _build_prompt(self, query, retrieved):
        context = "\n\n---\n\n".join([
            f"### Chunk {i+1} (score: {r['score']:.2f})\n{r['text']}"
            for i, r in enumerate(retrieved)
        ])
        # Llama 3 Instruct chat template format
        return (
            "<|begin_of_text|>"
            "<|start_header_id|>system<|end_header_id|>\n"
            "You are an expert technical documentation writer. "
            "Answer the user's question using ONLY the code context provided. "
            "Format your answer in clear Markdown with: function/class signatures, "
            "parameter descriptions with types, return value, and a usage example. "
            "Be precise and professional.<|eot_id|>"
            "<|start_header_id|>user<|end_header_id|>\n"
            f"=== CODE CONTEXT ===\n{context}\n\n"
            f"=== QUESTION ===\n{query}<|eot_id|>"
            "<|start_header_id|>assistant<|end_header_id|>\n"
        )

    def generate_documentation(self, query: str, top_k: int = 4) -> dict:
        retrieved = self.vs.search(query, top_k=top_k)
        if not retrieved:
            return {'answer': '❌ No relevant code found. Please upload your code files first!', 'sources': []}
        prompt = self._build_prompt(query, retrieved)
        if self.llm:
            raw = self.llm(prompt)[0]['generated_text']
            answer = raw.strip()
        else:
            answer = hf_inference_generate(prompt) or self._template_fallback(query, retrieved)
        sources = [
            f"`{r['chunk'].name}` in `{r['chunk'].file_path}` line {r['chunk'].line_number}"
            for r in retrieved
        ]
        return {'answer': answer, 'sources': sources, 'retrieved': retrieved}

    def _template_fallback(self, query, retrieved):
        lines = [f"## 📖 Documentation\n", f"**Query:** {query}\n"]
        for r in retrieved[:3]:
            c = r['chunk']
            lines.append(f"### `{c.name}` ({c.chunk_type})")
            if c.parent_class:
                lines.append(f"**Class:** `{c.parent_class}`")
            if c.args:
                lines.append(f"**Parameters:** {', '.join(f'`{a}`' for a in c.args)}")
            if c.docstring:
                lines.append(f"**Description:** {c.docstring}")
            lines.append(f"**Location:** `{c.file_path}` line {c.line_number}")
            lines.append(f"\n```python\n{c.source_code}\n```\n")
        lines.append("\n> 💡 Add HF_TOKEN to Kaggle Secrets for LLM-powered answers.")
        return "\n".join(lines)


assistant = TechDocAssistant(vector_store, llm_pipeline)
print("✅ Assistant ready!")

✅ Assistant ready!


## Phase 7: Gradio UI — with File Upload Tab

In [30]:
import gradio as gr
import shutil

UPLOAD_DIR = "uploaded_project"
os.makedirs(UPLOAD_DIR, exist_ok=True)

# ── Helper: rebuild index from a directory ──────────────────────────────────
def rebuild_index_from_dir(directory: str):
    new_chunks = parser.parse_directory(directory)
    if not new_chunks:
        return None, 0
    vector_store.build_index(new_chunks)
    assistant.vs = vector_store
    return new_chunks, len(new_chunks)


# ── Tab: Upload Your Code ───────────────────────────────────────────────────
def handle_upload(files):
    """Saves uploaded .py files and rebuilds the FAISS index."""
    if not files:
        return "⚠️ No files selected.", ""

    # Clear previous uploads
    shutil.rmtree(UPLOAD_DIR, ignore_errors=True)
    os.makedirs(UPLOAD_DIR, exist_ok=True)

    saved = []
    skipped = []
    for file in files:
        fname = os.path.basename(file.name)
        if fname.endswith('.py'):
            dest = os.path.join(UPLOAD_DIR, fname)
            shutil.copy(file.name, dest)
            saved.append(fname)
        else:
            skipped.append(fname)

    if not saved:
        return "❌ No `.py` files found. Please upload Python files only.", ""

    # Rebuild index
    new_chunks, total = rebuild_index_from_dir(UPLOAD_DIR)
    if not new_chunks:
        return "⚠️ Files uploaded but no functions/classes found. Check your code.", ""

    # Build summary
    status = f"✅ Uploaded & indexed {len(saved)} file(s) → {total} code chunks found!\n"
    if skipped:
        status += f"⚠️ Skipped (not .py): {', '.join(skipped)}\n"

    summary_lines = ["### 📂 Indexed from your files:\n"]
    for c in new_chunks:
        parent = f" → `{c.parent_class}`" if c.parent_class else ""
        summary_lines.append(
            f"- **[{c.chunk_type.upper()}]** `{c.name}`{parent} | `{c.file_path}:{c.line_number}`"
        )
    return status, "\n".join(summary_lines)


def use_sample_project():
    """Resets back to the built-in sample project."""
    new_chunks, total = rebuild_index_from_dir('sample_project')
    summary_lines = ["### 📂 Loaded sample project:\n"]
    for c in new_chunks:
        parent = f" → `{c.parent_class}`" if c.parent_class else ""
        summary_lines.append(
            f"- **[{c.chunk_type.upper()}]** `{c.name}`{parent} | `{c.file_path}:{c.line_number}`"
        )
    return f"✅ Reset to sample project → {total} chunks loaded.", "\n".join(summary_lines)


# ── Tab: Ask About Code ─────────────────────────────────────────────────────
def chat(message, history):
    if not message.strip():
        return history, ""
    result = assistant.generate_documentation(message)
    sources_md = ""
    if result['sources']:
        sources_md = "\n\n---\n**📎 Sources:**\n" + "\n".join(f"- {s}" for s in result['sources'])
    history.append((message, result['answer'] + sources_md))
    return history, ""


# ── Tab: Search Code Chunks ─────────────────────────────────────────────────
def search_code(query):
    if not query.strip():
        return "Enter a search query."
    results = vector_store.search(query, top_k=5)
    if not results:
        return "❌ No results found. Upload your code files first!"
    lines = [f"## 🔍 Top {len(results)} Results for: '{query}'\n"]
    for i, r in enumerate(results):
        c = r['chunk']
        lines.append(f"### {i+1}. `{c.name}` — Score: {r['score']:.3f}")
        lines.append(f"- **Type:** {c.chunk_type} | **File:** `{c.file_path}:{c.line_number}`")
        if c.parent_class:
            lines.append(f"- **Class:** `{c.parent_class}`")
        lines.append(f"```python\n{c.source_code[:400]}\n```\n")
    return "\n".join(lines)


# ── Build Gradio App ────────────────────────────────────────────────────────
with gr.Blocks(theme=gr.themes.Soft(), title="📚 Tech Doc AI Assistant v2") as demo:

    gr.Markdown("""
    # Project Technical Documentation AI Assistant
    **Upload your own Python files OR use the built-in sample project — powered by RAG + Llama 3**
    """)

    with gr.Tabs():

        # ── TAB 0: UPLOAD YOUR CODE (NEW!) ──────────────────────────────────
        with gr.TabItem("📤 Upload Your Code"):
            gr.Markdown("""
            ### Upload your own `.py` files here
            The assistant will index your code and answer questions about **your** project.
            You can upload **multiple files** at once.
            """)

            with gr.Row():
                file_upload = gr.File(
                    label="📂 Select your .py files",
                    file_types=[".py"],
                    file_count="multiple",
                    scale=4
                )

            with gr.Row():
                upload_btn = gr.Button("🚀 Upload & Index My Code", variant="primary", scale=2)
                sample_btn = gr.Button("🔄 Reset to Sample Project", variant="secondary", scale=1)

            upload_status = gr.Textbox(label="Status", interactive=False, lines=2)
            upload_summary = gr.Markdown(label="Indexed Chunks")

            upload_btn.click(handle_upload, inputs=[file_upload], outputs=[upload_status, upload_summary])
            sample_btn.click(use_sample_project, inputs=[], outputs=[upload_status, upload_summary])

            gr.Markdown("""
            ---
            > **Tip:** After uploading, go to the **💬 Ask About Code** tab and ask questions about your project!
            """)

        # ── TAB 1: CHAT ──────────────────────────────────────────────────────
        with gr.TabItem("💬 Ask About Code"):
            chatbot = gr.Chatbot(label="Documentation Assistant", height=420, bubble_full_width=False)
            with gr.Row():
                msg_input = gr.Textbox(
                    placeholder="e.g. How does the login function work? What does DatabaseManager do?",
                    label="Your Question", scale=5
                )
                submit_btn = gr.Button("Ask 🚀", variant="primary", scale=1)

            gr.Examples(
                examples=[
                    "How does the login function work?",
                    "Explain the DatabaseManager class",
                    "How is JWT token generated and verified?",
                    "What does the require_auth decorator do?",
                    "How does paginate_results work?",
                    "How is the password hashed before storing?"
                ],
                inputs=msg_input,
                label="💡 Example Questions (for sample project)"
            )
            clear_btn = gr.Button("🗑️ Clear Chat")
            submit_btn.click(chat, [msg_input, chatbot], [chatbot, msg_input])
            msg_input.submit(chat, [msg_input, chatbot], [chatbot, msg_input])
            clear_btn.click(lambda: ([], ""), None, [chatbot, msg_input])

        # ── TAB 2: VECTOR SEARCH ─────────────────────────────────────────────
        with gr.TabItem("🔍 Search Code Chunks"):
            gr.Markdown("Explore the raw FAISS semantic search results from your indexed codebase.")
            search_input = gr.Textbox(placeholder="e.g. database connection, token validation", label="Search Query")
            search_btn = gr.Button("Search 🔎", variant="secondary")
            search_output = gr.Markdown()
            search_btn.click(search_code, search_input, search_output)
            search_input.submit(search_code, search_input, search_output)

        # ── TAB 3: INDEXED CODEBASE ──────────────────────────────────────────
        with gr.TabItem("📂 Indexed Codebase"):
            def get_overview():
                if not vector_store.chunks:
                    return "No code indexed yet. Upload files or use the sample project."
                lines = ["## 📂 Currently Indexed Chunks\n"]
                for c in vector_store.chunks:
                    parent = f" → `{c.parent_class}`" if c.parent_class else ""
                    doc = f" — _{c.docstring[:60]}..._" if c.docstring else ""
                    lines.append(f"- **[{c.chunk_type.upper()}]** `{c.name}`{parent} | `{c.file_path}:{c.line_number}`{doc}")
                return "\n".join(lines)

            refresh_btn = gr.Button("🔄 Refresh List")
            overview_md = gr.Markdown(get_overview())
            refresh_btn.click(get_overview, None, overview_md)


print("🚀 Launching Gradio app...")
demo.launch(share=True, debug=False)

/tmp/ipykernel_57/4035906753.py:103: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), title="📚 Tech Doc AI Assistant v2") as demo:
/tmp/ipykernel_57/4035906753.py:145: UserWarning: You have not specified a value for the `type` parameter. Defaulting to the 'tuples' format for chatbot messages, but this is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style dictionaries with 'role' and 'content' keys.
  chatbot = gr.Chatbot(label="Documentation Assistant", height=420, bubble_full_width=False)
/tmp/ipykernel_57/4035906753.py:145: DeprecationWarning: The 'bubble_full_width' parameter will be removed in Gradio 6.0. This parameter no longer has any effect.
  chatbot = gr.Chatbot(label="Documentation Assistant", height=420, bubble_full_width=False)
/tmp/ipykernel_57/4035906753.

🚀 Launching Gradio app...
* Running on local URL:  http://127.0.0.1:7867
* Running on public URL: https://883daff519110d4d5b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


ERROR:    Exception in ASGI application
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/protocols/http/httptools_impl.py", line 416, in run_asgi
    result = await app(  # type: ignore[func-returns-value]
             ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/uvicorn/middleware/proxy_headers.py", line 60, in __call__
    return await self.app(scope, receive, send)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/fastapi/applications.py", line 1160, in __call__
    await super().__call__(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/applications.py", line 107, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/errors.py", line 186, in __call__
    raise exc
  File "/usr/local/lib/python3.12/dist-packages/starlette/middleware/error

⏳ Embedding 15 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS index built with 15 vectors!


Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


## Phase 8: Auto-Generate README from Current Index

In [34]:
import ast as ast_check
import re
from collections import defaultdict

# ── Step 1: Smarter docstring insertion ──
def get_docstring_for_chunk(chunk_name, chunk_code):
    """Ask LLM for just a one-line docstring, not the full file."""
    prompt = (
        "<|begin_of_text|>"
        "<|start_header_id|>system<|end_header_id|>\n"
        "You are a Python documentation expert. "
        "Reply with ONLY a single one-line docstring text — no quotes, no code, no explanation. "
        "Example reply: Trains the model for one epoch and saves checkpoints."
        "<|eot_id|>"
        "<|start_header_id|>user<|end_header_id|>\n"
        f"Write a one-line docstring for this Python function/class:\n\n{chunk_code[:800]}<|eot_id|>"
        "<|start_header_id|>assistant<|end_header_id|>\n"
    )
    if llm_pipeline:
        result = llm_pipeline(prompt)[0]['generated_text'].strip()
    else:
        result = hf_inference_generate(prompt) or ""
    # Clean up — take only first line, strip quotes
    result = result.splitlines()[0].strip().strip('"').strip("'")
    return result


def insert_docstrings_into_file(filepath):
    """Parses file, generates docstrings per chunk, inserts them, saves."""
    with open(filepath, 'r') as f:
        source = f.read()

    try:
        tree = ast_check.parse(source)
    except SyntaxError:
        print(f"   ⚠️ Skipping {filepath} — syntax error in original file")
        return source

    lines = source.splitlines()

    # Collect all functions/classes that need docstrings, in reverse order
    # (reverse so inserting lines doesn't shift line numbers)
    nodes_to_doc = []
    for node in ast_check.walk(tree):
        if isinstance(node, (ast_check.FunctionDef, ast_check.ClassDef)):
            existing_doc = ast_check.get_docstring(node)
            if not existing_doc:
                nodes_to_doc.append(node)

    # Sort by line number descending so insertions don't shift positions
    nodes_to_doc.sort(key=lambda n: n.lineno, reverse=True)

    for node in nodes_to_doc:
        # Get the code snippet for context (first 20 lines of the node)
        start = node.lineno - 1
        end = min(node.lineno + 20, len(lines))
        snippet = "\n".join(lines[start:end])

        docstring = get_docstring_for_chunk(node.name, snippet)
        if not docstring:
            continue

        # Find the line after def/class header (may span multiple lines)
        insert_line = node.lineno  # 1-indexed
        # Move past decorator lines
        body_start = node.body[0].lineno - 1  # 0-indexed
        # Detect indentation from first line of body
        body_line = lines[body_start]
        indent = len(body_line) - len(body_line.lstrip())
        indent_str = " " * indent

        docstring_line = f'{indent_str}"""{docstring}"""'
        lines.insert(body_start, docstring_line)

    result = "\n".join(lines)
    out_path = filepath.replace('uploaded_project', 'documented_project')
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    with open(out_path, 'w') as f:
        f.write(result)
    return result


os.makedirs('documented_project', exist_ok=True)
files = [f for f in os.listdir('uploaded_project') if f.endswith('.py')]
print(f"📂 Found {len(files)} files to document...\n")

for i, fname in enumerate(files, 1):
    filepath = f'uploaded_project/{fname}'
    chunks_in_file = [c for c in chunks if c.file_path == filepath]
    total_chunks = len([n for n in ast_check.walk(ast_check.parse(open(filepath).read()))
                        if isinstance(n, (ast_check.FunctionDef, ast_check.ClassDef))])
    print(f"⏳ [{i}/{len(files)}] {fname} — {total_chunks} functions/classes to document...")
    insert_docstrings_into_file(filepath)
    print(f"✅ [{i}/{len(files)}] Done: {fname}\n")

print("🎉 All files documented!")

# ── Step 2: Re-index documented files ──
new_chunks, total = rebuild_index_from_dir('documented_project')
print(f"✅ Re-indexed {total} chunks!")

# ── Step 3: Generate README ──
def generate_full_readme(chunks):
    by_file = defaultdict(list)
    for c in chunks:
        by_file[c.file_path].append(c)
    lines = [
        "# 📖 Auto-Generated Project Documentation",
        "> Generated by Technical Documentation AI Assistant\n",
        "## 📁 Project Structure\n"
    ]
    for filepath, file_chunks in by_file.items():
        lines.append(f"### `{filepath}`\n")
        classes = [c for c in file_chunks if c.chunk_type == 'class']
        functions = [c for c in file_chunks if c.chunk_type == 'function']
        if classes:
            lines.append("**Classes:**")
            for c in classes:
                lines.append(f"- `{c.name}` — {c.docstring or 'No description'}")
            lines.append("")
        if functions:
            lines.append("**Functions:**")
            for c in functions:
                args_str = ", ".join(c.args)
                lines.append(f"- `{c.name}({args_str})` — {c.docstring or 'No description'}")
            lines.append("")
        for cls in classes:
            methods = [c for c in file_chunks if c.parent_class == cls.name]
            if methods:
                lines.append(f"**`{cls.name}` Methods:**")
                for m in methods:
                    lines.append(f"| `{m.name}({', '.join(m.args)})` | {m.docstring or '—'} |")
                lines.append("")
    readme = "\n".join(lines)
    with open('AUTO_README.md', 'w') as f:
        f.write(readme)
    return readme

readme = generate_full_readme(vector_store.chunks)
print(readme)
print("\n💾 Saved to /kaggle/working/AUTO_README.md")

Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


📂 Found 2 files to document...

⏳ [1/2] train.py — 1 functions/classes to document...


Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✅ [1/2] Done: train.py

⏳ [2/2] model.py — 8 functions/classes to document...


Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=4096) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generati

✅ [2/2] Done: model.py

🎉 All files documented!
⚠️ Syntax error in documented_project/inference.py: invalid syntax (<unknown>, line 1)
⏳ Embedding 15 chunks...


Batches:   0%|          | 0/1 [00:00<?, ?it/s]

✅ FAISS index built with 15 vectors!
✅ Re-indexed 15 chunks!
# 📖 Auto-Generated Project Documentation
> Generated by Technical Documentation AI Assistant

## 📁 Project Structure

### `documented_project/train.py`

**Functions:**
- `train(resume_phase2_epoch)` — Trains the full RoBERTa model on the GoEmotions dataset and saves intermediate results.

### `documented_project/model.py`

**Classes:**
- `EmotionGatedCrossAttention` — Calculates the fused representation of input sequences and emotions using cross-attention and gated mechanisms.
- `DecoupledEmpatheticModel` — Initializes a decoupled empathetic model combining RoBERTa-based emotion classification and BART-based context encoding and conditional generation.

**Functions:**
- `__init__(hidden_size)` — Calculates the fused representation of input sequences and emotions using cross-attention and gated mechanisms.
- `forward(H, e_emotion)` — Calculates the fused representation by applying attention and gating mechanisms on input sequ

## How to Use Your Own Code

1. Run all cells
2. Open the Gradio public URL
3. Go to **Upload Your Code** tab
4. Click **Select your .py files** → pick your Python files
5. Click **Upload & Index My Code**
6. Go to **Ask About Code** and ask anything!

| Feature | Detail |
|---|---|
| **Upload multiple files** | Yes, all at once |
| **Reset to sample** | Click 'Reset to Sample Project' button |
| **Supported files** | `.py` Python files only |
| **Index updates live** | Yes, chat tab reflects new files instantly |